# Used Car Price Analytics & Machine Learning
## Car_Price_Analytics.ipynb
### Primary analytical notebook: Data Cleaning, EDA, Statistical Analysis, and ML Regression

**Dataset:** car_prices.csv — Used vehicle auction/sales transactions

**Business Problem:** Analyzing used vehicle sales patterns and predicting the selling price of used vehicles.

**Target Variable:** `sellingprice` (Regression)

## 0. Setup — Imports and Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import json
import os
import re
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

# ── Output directories ──────────────────────────────────────────────────
for d in ['../outputs/charts','../outputs/metrics','../outputs/predictions','../outputs/results',
          '../Node.js_UI/public/charts']:
    os.makedirs(d, exist_ok=True)

# ── Configurable model selection ─────────────────────────────────────────
# Change this to: 'linear_regression', 'ridge', 'random_forest', 'gradient_boosting'
MODEL_NAME = 'random_forest'

print('Libraries loaded. Model configured:', MODEL_NAME)

## 1. Dataset Inspection

In [ ]:
# Load raw dataset — keep a pristine copy
RAW_PATH = '../car_prices.csv'
df_raw = pd.read_csv(RAW_PATH)

print('=== SHAPE ===')
print(f'Rows: {df_raw.shape[0]:,}  |  Columns: {df_raw.shape[1]}')
print()
print('=== COLUMN NAMES ===')
print(df_raw.columns.tolist())
print()
print('=== DATA TYPES ===')
print(df_raw.dtypes)
print()
print('=== FIRST 3 ROWS ===')
df_raw.head(3)

In [ ]:
print('=== MISSING VALUES ===')
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0].to_string())
print()
print('=== DUPLICATE ROWS ===')
print(f'Duplicate rows: {df_raw.duplicated().sum():,}')
print()
print('=== DESCRIPTIVE STATISTICS ===')
df_raw.describe()

In [ ]:
print('=== UNIQUE VALUES PER CATEGORICAL COLUMN ===')
cat_cols = ['make','model','trim','body','transmission','state','color','interior','seller']
for c in cat_cols:
    print(f'{c}: {df_raw[c].nunique()} unique values')

print()
print('=== NUMERICAL COLUMNS ===')
print(['year','condition','odometer','mmr','sellingprice'])
print()
print('=== DATE/TIME COLUMNS ===')
print(['saledate'])
print()
print('=== IDENTIFIER COLUMNS ===')
print(['vin'])
print()
print('=== SALEDATE SAMPLE ===')
print(df_raw['saledate'].dropna().head(3).tolist())

## 2. Unit of Observation

**Each row represents a single used-vehicle auction/sale transaction.**

| Column | Meaning |
|---|---|
| `year` | Model year of the vehicle |
| `make` | Vehicle manufacturer (e.g., Ford, Toyota) |
| `model` | Vehicle model name (e.g., F-150, Camry) |
| `trim` | Trim level / sub-model variant |
| `body` | Body style (Sedan, SUV, Coupe, etc.) |
| `transmission` | Transmission type (automatic / manual) |
| `vin` | Vehicle Identification Number — unique vehicle identifier |
| `state` | US state where the auction/sale took place |
| `condition` | Condition score on a 1–49 scale (Manheim condition scale) |
| `odometer` | Odometer reading in miles at time of sale |
| `color` | Exterior colour of the vehicle |
| `interior` | Interior colour |
| `seller` | Name of the selling entity (dealer/fleet/finance company) |
| `mmr` | Manheim Market Report — an independent market valuation estimate (pre-sale reference price) |
| `sellingprice` | **TARGET** — actual transaction selling price (USD) |
| `saledate` | Date and time of the auction/sale event |

## 3. Data Cleaning

In [ ]:
df = df_raw.copy()
removal_log = []
start_rows = len(df)
print(f'Starting rows: {start_rows:,}')

In [ ]:
# ── 3.1 Drop rows with missing target (sellingprice) ─────────────────────
before = len(df)
df = df.dropna(subset=['sellingprice'])
removed = before - len(df)
removal_log.append({'Step': 'Drop missing sellingprice', 'Removed': removed})
print(f'Dropped {removed} rows with missing sellingprice. Remaining: {len(df):,}')

In [ ]:
# ── 3.2 Drop rows with missing saledate ──────────────────────────────────
before = len(df)
df = df.dropna(subset=['saledate'])
removed = before - len(df)
removal_log.append({'Step': 'Drop missing saledate', 'Removed': removed})
print(f'Dropped {removed} rows with missing saledate. Remaining: {len(df):,}')

In [ ]:
# ── 3.3 Parse saledate ───────────────────────────────────────────────────
# Format: 'Tue Dec 16 2014 12:30:00 GMT-0800 (PST)'
def parse_saledate(s):
    try:
        # Remove timezone abbreviation in parentheses
        s2 = re.sub(r'\(.*?\)', '', str(s)).strip()
        return pd.to_datetime(s2, format='%a %b %d %Y %H:%M:%S GMT%z', utc=True)
    except Exception:
        try:
            return pd.to_datetime(s, utc=True, errors='coerce')
        except:
            return pd.NaT

df['saledate_parsed'] = df['saledate'].apply(parse_saledate)
nat_count = df['saledate_parsed'].isna().sum()
print(f'Unparseable saledate rows: {nat_count}')
before = len(df)
df = df.dropna(subset=['saledate_parsed'])
removed = before - len(df)
removal_log.append({'Step': 'Drop unparseable saledate', 'Removed': removed})
print(f'Dropped {removed} rows with unparseable saledate. Remaining: {len(df):,}')

df['sale_year']    = df['saledate_parsed'].dt.year
df['sale_month']   = df['saledate_parsed'].dt.month
df['sale_quarter'] = df['saledate_parsed'].dt.quarter
df['sale_dow']     = df['saledate_parsed'].dt.dayofweek  # 0=Mon
print('saledate range:', df['saledate_parsed'].min(), 'to', df['saledate_parsed'].max())

In [ ]:
# ── 3.4 Invalid sellingprice ─────────────────────────────────────────────
before = len(df)
df = df[df['sellingprice'] >= 100]  # Prices below $100 are data-entry errors
removed = before - len(df)
removal_log.append({'Step': 'Drop sellingprice < $100 (invalid)', 'Removed': removed})
print(f'Dropped {removed} rows with sellingprice < $100. Remaining: {len(df):,}')

In [ ]:
# ── 3.5 Invalid year ─────────────────────────────────────────────────────
before = len(df)
df = df[(df['year'] >= 1980) & (df['year'] <= 2016)]
removed = before - len(df)
removal_log.append({'Step': 'Drop invalid year (outside 1980-2016)', 'Removed': removed})
print(f'Dropped {removed} rows with invalid year. Remaining: {len(df):,}')

In [ ]:
# ── 3.6 Invalid odometer ─────────────────────────────────────────────────
before = len(df)
df = df[(df['odometer'].isna()) | ((df['odometer'] >= 1) & (df['odometer'] <= 500000))]
removed = before - len(df)
removal_log.append({'Step': 'Drop odometer > 500,000 or < 1 (invalid)', 'Removed': removed})
print(f'Dropped {removed} rows with impossible odometer. Remaining: {len(df):,}')

In [ ]:
# ── 3.7 Fix transmission — remove bad values ('sedan','Sedan') ───────────
valid_trans = ['automatic', 'manual']
df['transmission'] = df['transmission'].str.lower().str.strip()
df.loc[~df['transmission'].isin(valid_trans), 'transmission'] = np.nan
print('Transmission value counts after fix:')
print(df['transmission'].value_counts(dropna=False))

In [ ]:
# ── 3.8 Standardize body style ───────────────────────────────────────────
df['body'] = df['body'].str.strip().str.title()
print('Top body styles after standardization:')
print(df['body'].value_counts().head(15))

In [ ]:
# ── 3.9 Fix interior encoding artifact ───────────────────────────────────
df['interior'] = df['interior'].str.strip()
# Replace garbled chars with NaN
df.loc[df['interior'].str.contains(r'[^\x00-\x7F]', na=False, regex=True), 'interior'] = np.nan
print('Interior value counts (top 10):')
print(df['interior'].value_counts().head(10))

In [ ]:
# ── 3.10 Fill missing categoricals with 'Unknown' ────────────────────────
for col in ['make','model','trim','body','transmission','color','interior']:
    before_na = df[col].isna().sum()
    df[col] = df[col].fillna('Unknown')
    print(f'{col}: filled {before_na} NaN → Unknown')

In [ ]:
# ── 3.11 Impute missing odometer with median ──────────────────────────────
odo_median = df['odometer'].median()
odo_na = df['odometer'].isna().sum()
df['odometer'] = df['odometer'].fillna(odo_median)
print(f'Imputed {odo_na} missing odometer values with median: {odo_median:,.0f}')

In [ ]:
# ── 3.12 Impute missing condition with median ─────────────────────────────
cond_median = df['condition'].median()
cond_na = df['condition'].isna().sum()
df['condition'] = df['condition'].fillna(cond_median)
print(f'Imputed {cond_na} missing condition values with median: {cond_median}')

In [ ]:
# ── 3.13 Impute missing MMR with median ───────────────────────────────────
mmr_median = df['mmr'].median()
mmr_na = df['mmr'].isna().sum()
df['mmr'] = df['mmr'].fillna(mmr_median)
print(f'Imputed {mmr_na} missing MMR values with median: {mmr_median:,.0f}')

In [ ]:
# ── 3.14 Removal log summary ──────────────────────────────────────────────
print('\n=== DATA CLEANING SUMMARY ===')
clean_df = pd.DataFrame(removal_log)
print(clean_df.to_string(index=False))
total_removed = start_rows - len(df)
print(f'\nTotal removed: {total_removed:,} ({total_removed/start_rows*100:.2f}% of original)')
print(f'Final clean dataset: {len(df):,} rows')

## 4. Outlier Investigation

In [ ]:
# Investigate outliers — do NOT blindly remove
for col in ['sellingprice','mmr','odometer','condition']:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lo = Q1 - 1.5*IQR
    hi = Q3 + 1.5*IQR
    n_out = ((df[col] < lo) | (df[col] > hi)).sum()
    print(f'{col}: IQR range [{lo:,.1f} – {hi:,.1f}]  |  IQR-outliers: {n_out:,} ({n_out/len(df)*100:.2f}%)')

print()
print('High-value vehicles (sellingprice > $80,000):')
print(df[df['sellingprice'] > 80000][['year','make','model','sellingprice','mmr']].head(10))
print()
print('Decision: Retain all records with sellingprice >= $100 and odometer <= 500,000.')
print('Expensive vehicles are legitimate luxury/exotic cars.')
print('High-mileage outliers beyond 500k were removed as data-entry errors.')

## 5. Feature Engineering

In [ ]:
# ── vehicle_age ───────────────────────────────────────────────────────────
df['vehicle_age'] = df['sale_year'] - df['year']
# Clamp to 0 (some 2015 model year cars sold in 2014 auctions)
df['vehicle_age'] = df['vehicle_age'].clip(lower=0)
print('vehicle_age stats:')
print(df['vehicle_age'].describe())

In [ ]:
# ── mileage_per_year ──────────────────────────────────────────────────────
df['mileage_per_year'] = df['odometer'] / (df['vehicle_age'].replace(0, 1))
print('mileage_per_year stats:')
print(df['mileage_per_year'].describe())

In [ ]:
# ── log transforms ────────────────────────────────────────────────────────
df['log_odometer'] = np.log1p(df['odometer'])
df['log_sellingprice'] = np.log1p(df['sellingprice'])

print('Feature engineering complete.')
print('New features: vehicle_age, mileage_per_year, log_odometer, log_sellingprice')
print()
print('NOTE: log_sellingprice is used for visualization only — NOT as an ML feature')
print('      to avoid target leakage. price_diff and price_ratio derived from')
print('      sellingprice are EXCLUDED from model features for the same reason.')
print()
print('sale_month and sale_quarter are available date features (no leakage).')

## 6. Target Leakage Check — MMR Analysis

**MMR (Manheim Market Report)** is an independent, pre-sale market valuation provided by Manheim auction houses **before** the vehicle is sold. It is calculated from historical transaction data for comparable vehicles and is used by dealers, fleet companies, and financial institutions as a reference price **before bidding or listing a vehicle**.

**Conclusion:** MMR is a valid pre-sale predictor. It is analogous to a Zillow Zestimate for homes — available before the sale occurs. Therefore:

- **Model A (With MMR):** Uses MMR as a feature → simulates a dealer/auctioneer who has access to the Manheim report before the sale.
- **Model B (Without MMR):** Excludes MMR → simulates predicting from vehicle characteristics alone.

Features derived **directly from sellingprice** (e.g., `sellingprice - mmr`, `sellingprice / mmr`) are **excluded** from all models — these would constitute target leakage.

## 7. Exploratory Data Analysis (EDA)

In [ ]:
print('=== EDA SUMMARY STATISTICS ===')
print(df[['sellingprice','mmr','odometer','condition','vehicle_age']].describe().round(2))

In [ ]:
print('=== TOP 10 MAKES BY VOLUME ===')
print(df['make'].value_counts().head(10).to_string())
print()
print('=== AVERAGE SELLING PRICE BY TOP 10 MAKES ===')
top_makes = df['make'].value_counts().head(10).index
avg_by_make = df[df['make'].isin(top_makes)].groupby('make')['sellingprice'].mean().sort_values(ascending=False)
print(avg_by_make.round(2).to_string())
print()
print('=== CORRELATION MATRIX (numerical) ===')
corr = df[['sellingprice','mmr','odometer','condition','vehicle_age','mileage_per_year']].corr()
print(corr.round(3).to_string())

## 8. Main Visualizations

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# MAIN VISUALIZATION 1 — Selling Price Distribution
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw distribution
axes[0].hist(df['sellingprice'], bins=80, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].set_title('Distribution of Selling Price (Raw)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Selling Price (USD)')
axes[0].set_ylabel('Count')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
mean_p = df['sellingprice'].mean()
median_p = df['sellingprice'].median()
axes[0].axvline(mean_p,   color='red',    linestyle='--', linewidth=1.5, label=f'Mean ${mean_p:,.0f}')
axes[0].axvline(median_p, color='orange', linestyle='--', linewidth=1.5, label=f'Median ${median_p:,.0f}')
axes[0].legend()

# Log-scale distribution
axes[1].hist(df['log_sellingprice'], bins=80, color='teal', edgecolor='white', linewidth=0.3)
axes[1].set_title('Distribution of Log(Selling Price)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Log(Selling Price)')
axes[1].set_ylabel('Count')

plt.suptitle('MAIN VISUALIZATION 1: Selling Price Distribution', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
for out in ['../outputs/charts/viz1_price_distribution.png', '../Node.js_UI/public/charts/viz1_price_distribution.png']:
    plt.savefig(out, bbox_inches='tight', dpi=120)
plt.show()
print(f'Mean selling price: ${mean_p:,.2f}')
print(f'Median selling price: ${median_p:,.2f}')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# MAIN VISUALIZATION 2 — Selling Price vs Vehicle Age
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
age_price = df.groupby('vehicle_age')['sellingprice'].median().reset_index()
age_price = age_price[age_price['vehicle_age'] <= 20]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(age_price['vehicle_age'], age_price['sellingprice'], marker='o', color='steelblue', linewidth=2, markersize=6)
ax.fill_between(age_price['vehicle_age'], age_price['sellingprice'], alpha=0.15, color='steelblue')
ax.set_title('MAIN VISUALIZATION 2: Median Selling Price by Vehicle Age', fontsize=14, fontweight='bold')
ax.set_xlabel('Vehicle Age (years at time of sale)')
ax.set_ylabel('Median Selling Price (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_xticks(age_price['vehicle_age'])
plt.tight_layout()
for out in ['../outputs/charts/viz2_price_vs_age.png', '../Node.js_UI/public/charts/viz2_price_vs_age.png']:
    plt.savefig(out, bbox_inches='tight', dpi=120)
plt.show()
print('Age-Price table (age 0-10):')
print(age_price[age_price['vehicle_age'] <= 10].to_string(index=False))

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# MAIN VISUALIZATION 3 — Selling Price vs Odometer
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Use a 2D hexbin to avoid overplotting on 550k+ rows
fig, ax = plt.subplots(figsize=(12, 5))
hb = ax.hexbin(df['odometer'], df['sellingprice'], gridsize=60,
               cmap='YlOrRd', mincnt=1, bins='log')
cb = plt.colorbar(hb, ax=ax)
cb.set_label('log10(count)')
ax.set_title('MAIN VISUALIZATION 3: Selling Price vs Odometer Reading', fontsize=14, fontweight='bold')
ax.set_xlabel('Odometer (miles)')
ax.set_ylabel('Selling Price (USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
for out in ['../outputs/charts/viz3_price_vs_odometer.png', '../Node.js_UI/public/charts/viz3_price_vs_odometer.png']:
    plt.savefig(out, bbox_inches='tight', dpi=120)
plt.show()
corr_odo = df['odometer'].corr(df['sellingprice'])
print(f'Pearson correlation odometer vs sellingprice: {corr_odo:.4f}')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# MAIN VISUALIZATION 4 — Average Selling Price by Top 10 Makes
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
top10_makes = df['make'].value_counts().head(10).index
make_avg = (df[df['make'].isin(top10_makes)]
            .groupby('make')['sellingprice']
            .mean()
            .sort_values(ascending=False)
            .reset_index())
make_avg.columns = ['make', 'avg_price']

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(make_avg['make'][::-1], make_avg['avg_price'][::-1],
               color=sns.color_palette('muted', len(make_avg)))
for bar, val in zip(bars, make_avg['avg_price'][::-1]):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
            f'${val:,.0f}', va='center', fontsize=9)
ax.set_title('MAIN VISUALIZATION 4: Average Selling Price — Top 10 Makes by Volume', fontsize=14, fontweight='bold')
ax.set_xlabel('Average Selling Price (USD)')
ax.set_ylabel('Make')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
for out in ['../outputs/charts/viz4_avg_price_by_make.png', '../Node.js_UI/public/charts/viz4_avg_price_by_make.png']:
    plt.savefig(out, bbox_inches='tight', dpi=120)
plt.show()
print(make_avg.to_string(index=False))

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# MAIN VISUALIZATION 5 — MMR vs Selling Price
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Sample 20,000 rows for scatter clarity
sample_mmr = df.sample(20000, random_state=42)

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(sample_mmr['mmr'], sample_mmr['sellingprice'],
           alpha=0.15, s=8, color='steelblue', label='Sale transaction')
lims = [0, min(df['mmr'].max(), df['sellingprice'].max()) * 1.02]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect match line')
ax.set_title('MAIN VISUALIZATION 5: MMR vs Actual Selling Price', fontsize=14, fontweight='bold')
ax.set_xlabel('MMR — Manheim Market Report Estimate (USD)')
ax.set_ylabel('Actual Selling Price (USD)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
plt.tight_layout()
for out in ['../outputs/charts/viz5_mmr_vs_price.png', '../Node.js_UI/public/charts/viz5_mmr_vs_price.png']:
    plt.savefig(out, bbox_inches='tight', dpi=120)
plt.show()
corr_mmr = df['mmr'].corr(df['sellingprice'])
print(f'Pearson correlation MMR vs sellingprice: {corr_mmr:.4f}')

## 9. Observations and Insights

In [ ]:
# Compute values needed for observations
obs_mean_price   = df['sellingprice'].mean()
obs_median_price = df['sellingprice'].median()
obs_corr_odo     = df['odometer'].corr(df['sellingprice'])
obs_corr_mmr     = df['mmr'].corr(df['sellingprice'])
obs_corr_age     = df['vehicle_age'].corr(df['sellingprice'])
obs_top_make     = make_avg.iloc[0]['make']
obs_top_make_avg = make_avg.iloc[0]['avg_price']
obs_low_make     = make_avg.iloc[-1]['make']
obs_low_make_avg = make_avg.iloc[-1]['avg_price']
obs_pct_auto     = (df['transmission'] == 'automatic').sum() / len(df) * 100
obs_age0_price   = age_price[age_price['vehicle_age'] == 0]['sellingprice'].values[0] if 0 in age_price['vehicle_age'].values else None
obs_age10_price  = age_price[age_price['vehicle_age'] == 10]['sellingprice'].values[0] if 10 in age_price['vehicle_age'].values else None

print('Values computed for observations.')

In [ ]:
print('=' * 70)
print('EXACTLY 5 OBSERVATIONS')
print('=' * 70)
print()
print(f'Observation 1:')
print(f'  The selling price distribution is right-skewed with a mean of')
print(f'  ${obs_mean_price:,.2f} and a median of ${obs_median_price:,.2f}, indicating that a')
print(f'  minority of high-value vehicles pulls the mean above the median.')
print()
print(f'Observation 2:')
print(f'  Vehicle age has a negative correlation of {obs_corr_age:.4f} with selling')
print(f'  price. Median selling price for age-0 vehicles (${obs_age0_price:,.0f}) is')
print(f'  substantially higher than age-10 vehicles (${obs_age10_price:,.0f}), confirming')
print(f'  a consistent depreciation trend.')
print()
print(f'Observation 3:')
print(f'  Odometer reading has a negative Pearson correlation of {obs_corr_odo:.4f}')
print(f'  with selling price, indicating that higher mileage is associated with')
print(f'  lower transaction prices.')
print()
print(f'Observation 4:')
print(f'  Among the top 10 most frequently sold makes, {obs_top_make} commands the')
print(f'  highest average selling price (${obs_top_make_avg:,.0f}), while {obs_low_make}')
print(f'  records the lowest average (${obs_low_make_avg:,.0f}) within this group.')
print()
print(f'Observation 5:')
print(f'  MMR (Manheim Market Report) has a very strong Pearson correlation of')
print(f'  {obs_corr_mmr:.4f} with actual selling price, suggesting that the pre-sale')
print(f'  market estimate is a powerful indicator of final transaction value.')

In [ ]:
print('=' * 70)
print('EXACTLY 5 INSIGHTS')
print('=' * 70)
print()
print('Insight 1:')
print('  OBSERVATION: The selling price distribution is right-skewed.')
print('  INSIGHT: The majority of transactions involve lower-priced vehicles,')
print('  but a tail of high-value sales inflates the mean. Pricing strategies')
print('  should be calibrated to the median rather than the mean to represent')
print('  the typical market transaction.')
print()
print('Insight 2:')
print(f'  OBSERVATION: Median price drops from ${obs_age0_price:,.0f} for new cars to')
print(f'  ${obs_age10_price:,.0f} at 10 years of age.')
print('  INSIGHT: Depreciation is steepest in early years, which means')
print('  near-new used vehicles still command premium prices. Dealers')
print('  sourcing 1-3 year old vehicles can capture the depreciation')
print('  benefit while still achieving high resale values.')
print()
print('Insight 3:')
print(f'  OBSERVATION: Odometer correlation with selling price is {obs_corr_odo:.3f}.')
print('  INSIGHT: Mileage is a meaningful negative predictor of price, but the')
print('  moderate correlation suggests condition and vehicle age also play')
print('  important roles. High-mileage luxury vehicles can still sell above')
print('  low-mileage economy cars.')
print()
print('Insight 4:')
print(f'  OBSERVATION: {obs_top_make} achieves the highest average selling price')
print(f'  among the top-10 volume makes at ${obs_top_make_avg:,.0f}.')
print('  INSIGHT: Brand premium persists in the used-car market. Dealers')
print('  and fleet managers should factor brand positioning when setting')
print('  reserve prices at auction.')
print()
print('Insight 5:')
print(f'  OBSERVATION: MMR correlates at {obs_corr_mmr:.3f} with actual selling price.')
print('  INSIGHT: The Manheim pre-sale valuation is highly predictive of actual')
print('  transaction outcomes. Including MMR as a feature in a pricing model')
print('  substantially improves prediction accuracy — this provides strong')
print('  justification for using MMR-assisted valuation in real-time pricing tools.')

## 10. Hypotheses and Statistical Testing

In [ ]:
alpha = 0.05
print('Statistical significance level (alpha):', alpha)
print()
print('=' * 70)
print('HYPOTHESIS 1: Average selling prices differ across vehicle makes')
print('(ANOVA / Kruskal-Wallis)')
print('=' * 70)
print()
print('H0: The mean selling price is equal across the top 5 vehicle makes.')
print('H1: At least one make has a significantly different mean selling price.')
print()
# Kruskal-Wallis (non-parametric, appropriate for skewed data)
top5_makes = df['make'].value_counts().head(5).index.tolist()
groups = [df[df['make'] == m]['sellingprice'].values for m in top5_makes]
kw_stat, kw_p = stats.kruskal(*groups)
print(f'Test: Kruskal-Wallis H-test (used because selling price is right-skewed)')
print(f'Groups tested: {top5_makes}')
print(f'H-statistic: {kw_stat:,.4f}')
print(f'p-value: {kw_p:.6e}')
print(f'Decision: {"Reject H0" if kw_p < alpha else "Fail to reject H0"} at alpha={alpha}')
print(f'Conclusion: Selling prices differ significantly across makes.')
print(f'Business interpretation: Make is a meaningful pricing factor; different')
print(f'makes cannot be priced using a single uniform rule.')

In [ ]:
print('=' * 70)
print('HYPOTHESIS 2: Vehicle mileage is statistically associated with selling price')
print('(Spearman Rank Correlation)')
print('=' * 70)
print()
print('H0: There is no monotonic association between odometer and selling price (rho = 0).')
print('H1: There is a significant monotonic association between odometer and selling price.')
print()
# Use a sample for speed (Spearman on 550k rows is slow)
sample_h2 = df[['odometer','sellingprice']].sample(30000, random_state=42)
sp_r, sp_p = stats.spearmanr(sample_h2['odometer'], sample_h2['sellingprice'])
print(f'Test: Spearman rank correlation (monotonic association, no normality assumed)')
print(f'Sample size: 30,000')
print(f'Spearman rho: {sp_r:.4f}')
print(f'p-value: {sp_p:.6e}')
print(f'Decision: {"Reject H0" if sp_p < alpha else "Fail to reject H0"} at alpha={alpha}')
print(f'Conclusion: Odometer reading has a significant negative association with selling price.')
print(f'Business interpretation: Higher mileage vehicles are valued lower; mileage')
print(f'should be a key input in any used-car valuation tool.')

In [ ]:
print('=' * 70)
print('HYPOTHESIS 3: MMR is statistically associated with actual selling price')
print('(Pearson Correlation)')
print('=' * 70)
print()
print('H0: There is no linear association between MMR and selling price (r = 0).')
print('H1: There is a significant linear association between MMR and selling price.')
print()
sample_h3 = df[['mmr','sellingprice']].sample(30000, random_state=42)
pr_r, pr_p = stats.pearsonr(sample_h3['mmr'], sample_h3['sellingprice'])
print(f'Test: Pearson product-moment correlation (linear relationship)')
print(f'Sample size: 30,000')
print(f'Pearson r: {pr_r:.4f}')
print(f'p-value: {pr_p:.6e}')
print(f'Decision: {"Reject H0" if pr_p < alpha else "Fail to reject H0"} at alpha={alpha}')
print(f'Conclusion: MMR has a very strong, statistically significant positive')
print(f'linear association with actual selling price.')
print(f'Business interpretation: The Manheim pre-sale estimate is highly reliable')
print(f'as a pricing anchor. Incorporating MMR into valuation workflows can')
print(f'substantially improve pricing accuracy.')
print()
print('NOTE: Statistical significance does not imply causation. These are')
print('associative findings. Practical significance also depends on effect size.')

## 11. Machine Learning — Price Prediction (Regression)

In [ ]:
# ── Feature sets ──────────────────────────────────────────────────────────
# Features that do NOT require MMR (vehicle-characteristics only)
BASE_NUM_FEATURES   = ['year', 'condition', 'odometer', 'vehicle_age', 'mileage_per_year',
                        'sale_month', 'sale_quarter']
BASE_CAT_FEATURES   = ['make', 'body', 'transmission', 'state']
BASE_FEATURES       = BASE_NUM_FEATURES + BASE_CAT_FEATURES

# Features that include MMR (with Manheim pre-sale estimate)
MMR_NUM_FEATURES    = BASE_NUM_FEATURES + ['mmr']
MMR_CAT_FEATURES    = BASE_CAT_FEATURES
MMR_FEATURES        = MMR_NUM_FEATURES + MMR_CAT_FEATURES

TARGET = 'sellingprice'

print('TARGET:', TARGET)
print('Model A (with MMR) features:', MMR_FEATURES)
print()
print('Model B (without MMR) features:', BASE_FEATURES)

In [ ]:
# ── Train/test split — random (general prediction, not temporal) ──────────
# Rationale: Goal is general price prediction, not time-series forecasting.
# A random 80/20 split produces a representative test set.
X_full = df[MMR_FEATURES].copy()
y = df[TARGET].copy()

X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_full, y, test_size=0.20, random_state=42
)

# Also create base (no-MMR) splits
X_base = df[BASE_FEATURES].copy()
X_train_base = X_base.loc[X_train_full.index]
X_test_base  = X_base.loc[X_test_full.index]

print(f'Train size: {len(X_train_full):,}  |  Test size: {len(X_test_full):,}')
print(f'Train %: {len(X_train_full)/len(df)*100:.1f}%  |  Test %: {len(X_test_full)/len(df)*100:.1f}%')

In [ ]:
# ── Build preprocessing pipelines ────────────────────────────────────────
def build_preprocessor(num_features, cat_features):
    num_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler())
    ])
    cat_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    return ColumnTransformer([
        ('num', num_pipe, num_features),
        ('cat', cat_pipe, cat_features)
    ])

pre_mmr  = build_preprocessor(MMR_NUM_FEATURES,  MMR_CAT_FEATURES)
pre_base = build_preprocessor(BASE_NUM_FEATURES, BASE_CAT_FEATURES)
print('Preprocessing pipelines built.')

In [ ]:
# ── Model registry ────────────────────────────────────────────────────────
MODEL_REGISTRY = {
    'linear_regression': LinearRegression(),
    'ridge':             Ridge(alpha=1.0),
    'random_forest':     RandomForestRegressor(n_estimators=100, max_depth=20,
                                               min_samples_leaf=5, n_jobs=-1, random_state=42),
    'gradient_boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5,
                                                    learning_rate=0.1, random_state=42,
                                                    subsample=0.8)
}

chosen_estimator = MODEL_REGISTRY[MODEL_NAME]
print(f'Selected model: {MODEL_NAME}')
print(chosen_estimator)

In [ ]:
# ── Train Model A (with MMR) ───────────────────────────────────────────────
import copy
model_A = Pipeline([
    ('preprocessor', pre_mmr),
    ('regressor',    copy.deepcopy(chosen_estimator))
])
print('Training Model A (with MMR)...')
model_A.fit(X_train_full, y_train)
print('Model A training complete.')

In [ ]:
# ── Train Model B (without MMR) ───────────────────────────────────────────
model_B = Pipeline([
    ('preprocessor', pre_base),
    ('regressor',    copy.deepcopy(chosen_estimator))
])
print('Training Model B (without MMR)...')
model_B.fit(X_train_base, y_train)
print('Model B training complete.')

In [ ]:
# ── Evaluation helper ──────────────────────────────────────────────────────
def evaluate_model(name, model, X_test, y_test):
    preds = model.predict(X_test)
    mae   = mean_absolute_error(y_test, preds)
    mse   = mean_squared_error(y_test, preds)
    rmse  = np.sqrt(mse)
    r2    = r2_score(y_test, preds)
    med_ae = np.median(np.abs(y_test.values - preds))
    result = {
        'Model': name,
        'MAE':   round(mae, 2),
        'MSE':   round(mse, 2),
        'RMSE':  round(rmse, 2),
        'R2':    round(r2, 4),
        'Median_AE': round(med_ae, 2)
    }
    print(f'\n=== {name} ===')
    print(f'  MAE:       ${mae:,.2f}')
    print(f'  RMSE:      ${rmse:,.2f}')
    print(f'  R²:        {r2:.4f}')
    print(f'  Median AE: ${med_ae:,.2f}')
    return result, preds

metrics_A, preds_A = evaluate_model(f'{MODEL_NAME.upper()} — WITH MMR (Model A)',    model_A, X_test_full, y_test)
metrics_B, preds_B = evaluate_model(f'{MODEL_NAME.upper()} — WITHOUT MMR (Model B)', model_B, X_test_base, y_test)

In [ ]:
# ── Model comparison table ─────────────────────────────────────────────────
comparison_df = pd.DataFrame([metrics_A, metrics_B])
print('\n=== MODEL COMPARISON TABLE ===')
print(comparison_df[['Model','MAE','RMSE','R2','Median_AE']].to_string(index=False))
print()
print('Selection criterion: Lowest RMSE + Highest R².')
best_idx = comparison_df['R2'].idxmax()
print(f'Best model (by R²): {comparison_df.loc[best_idx, "Model"]}')
print()
print('Metric explanations:')
print('  MAE  — Mean Absolute Error: average dollar error; easy to interpret')
print('  RMSE — Root Mean Squared Error: penalises large errors more than MAE')
print('  R²   — Coefficient of determination: proportion of variance explained')
print('  Median AE — robust to extreme outliers in prediction errors')

In [ ]:
# ── Save metrics to JSON for Node.js UI ───────────────────────────────────
metrics_export = {
    'model_name': MODEL_NAME,
    'model_A': metrics_A,
    'model_B': metrics_B,
    'comparison': comparison_df[['Model','MAE','RMSE','R2']].to_dict(orient='records')
}
for path in ['../outputs/metrics/model_metrics.json', '../Node.js_UI/public/model_metrics.json']:
    with open(path, 'w') as f:
        json.dump(metrics_export, f, indent=2)
print('Metrics saved to outputs/metrics/model_metrics.json and Node.js_UI/public/model_metrics.json')

## 12. Prediction Output

In [ ]:
# Generate prediction table from Model A (primary model)
pred_table = X_test_full[['mmr']].copy()
pred_table['actual_sellingprice']    = y_test.values
pred_table['predicted_sellingprice'] = preds_A.round(2)
pred_table['prediction_error']       = (pred_table['predicted_sellingprice'] - pred_table['actual_sellingprice']).round(2)
pred_table['abs_error']              = pred_table['prediction_error'].abs().round(2)
pred_table['pct_error']              = (pred_table['abs_error'] / pred_table['actual_sellingprice'].replace(0, np.nan) * 100).round(2)
pred_table = pred_table.reset_index(drop=True)

print('=== PREDICTION TABLE (first 20 rows) ===')
print(pred_table[['actual_sellingprice','predicted_sellingprice','prediction_error','abs_error','pct_error']].head(20).to_string(index=False))
print()
print(f'Mean absolute error (from table): ${pred_table["abs_error"].mean():,.2f}')
print(f'Median absolute error:            ${pred_table["abs_error"].median():,.2f}')
print(f'Mean percentage error (MAPE):      {pred_table["pct_error"].mean():.2f}%')

In [ ]:
# Save predictions CSV
pred_table.to_csv('../outputs/predictions/test_predictions.csv', index=False)
# Save sample for Node.js
pred_table.head(100).to_json('../Node.js_UI/public/predictions_sample.json', orient='records', indent=2)
print('Predictions saved.')

## 13. MMR / Valuation Analysis

In [ ]:
# Price difference analysis — for analytical insight, NOT as model features
df['price_diff'] = df['sellingprice'] - df['mmr']  # used for analysis only
df['price_ratio'] = df['sellingprice'] / df['mmr'].replace(0, np.nan)

# Define valuation categories (analytical rule, clearly labeled)
def valuation_category(ratio):
    if pd.isna(ratio):
        return 'Unknown'
    elif ratio > 1.05:
        return 'Sold Above MMR (>5%)'
    elif ratio < 0.95:
        return 'Sold Below MMR (<5%)'
    else:
        return 'Sold Near MMR (±5%)'

df['valuation_cat'] = df['price_ratio'].apply(valuation_category)

print('=== VALUATION CATEGORY DISTRIBUTION ===')
val_dist = df['valuation_cat'].value_counts()
print(val_dist)
print()
print(f'Average price diff (selling - MMR): ${df["price_diff"].mean():,.2f}')
print(f'Median price diff (selling - MMR):  ${df["price_diff"].median():,.2f}')

# Save valuation summary for Node.js
val_summary = {
    'avg_price_diff': round(float(df['price_diff'].mean()), 2),
    'median_price_diff': round(float(df['price_diff'].median()), 2),
    'valuation_distribution': val_dist.to_dict()
}
for path in ['../outputs/results/valuation_summary.json', '../Node.js_UI/public/valuation_summary.json']:
    with open(path, 'w') as f:
        json.dump(val_summary, f, indent=2)
print('Valuation summary saved.')

## 14. Model Explainability — Feature Importance

In [ ]:
# Extract feature importance from Model A regressor
regressor = model_A.named_steps['regressor']
preprocessor = model_A.named_steps['preprocessor']

if hasattr(regressor, 'feature_importances_'):
    importances = regressor.feature_importances_
    # Get feature names from ColumnTransformer
    num_names = MMR_NUM_FEATURES
    cat_encoder = preprocessor.named_transformers_['cat'].named_steps['encoder']
    cat_names = list(cat_encoder.get_feature_names_out(MMR_CAT_FEATURES))
    all_names = num_names + cat_names
    fi_df = pd.DataFrame({'feature': all_names, 'importance': importances})
    fi_df = fi_df.sort_values('importance', ascending=False).head(20)

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(fi_df['feature'][::-1], fi_df['importance'][::-1], color='steelblue')
    ax.set_title(f'Top 20 Feature Importances — Model A ({MODEL_NAME})', fontsize=13, fontweight='bold')
    ax.set_xlabel('Importance (Gini / impurity-based)')
    ax.set_ylabel('Feature')
    plt.tight_layout()
    for out in ['../outputs/charts/feature_importance.png', '../Node.js_UI/public/charts/feature_importance.png']:
        plt.savefig(out, bbox_inches='tight', dpi=120)
    plt.show()

    print('Top 10 features by importance:')
    print(fi_df.head(10).to_string(index=False))

    fi_export = fi_df.to_dict(orient='records')
    for path in ['../outputs/results/feature_importance.json', '../Node.js_UI/public/feature_importance.json']:
        with open(path, 'w') as f:
            json.dump(fi_export, f, indent=2)
    print('Feature importance saved.')
elif hasattr(regressor, 'coef_'):
    print('Linear model: coefficients used as importance.')
    num_names = MMR_NUM_FEATURES
    cat_encoder = preprocessor.named_transformers_['cat'].named_steps['encoder']
    cat_names = list(cat_encoder.get_feature_names_out(MMR_CAT_FEATURES))
    all_names = num_names + cat_names
    coef = regressor.coef_
    fi_df = pd.DataFrame({'feature': all_names, 'coefficient': coef})
    fi_df['abs_coef'] = fi_df['coefficient'].abs()
    fi_df = fi_df.sort_values('abs_coef', ascending=False).head(20)
    print(fi_df.to_string(index=False))
else:
    print('Feature importance not directly available for this model type.')

## 15. Exactly 3 Business Recommendations

In [ ]:
print('=' * 70)
print('EXACTLY 3 BUSINESS RECOMMENDATIONS')
print('=' * 70)
print()
r_mae  = metrics_A['MAE']
r_r2   = metrics_A['R2']
r_diff = val_summary['median_price_diff']

print('Recommendation 1: MMR-Assisted Pricing Workflow')
print(f'  The machine learning model (Model A) achieves an MAE of ${r_mae:,.2f}')
print(f'  and R² of {r_r2:.4f} when MMR is included. Given that the median')
print(f'  selling price deviation from MMR is ${abs(r_diff):,.2f}, dealers and')
print(f'  auction houses should integrate the Manheim Market Report into their')
print(f'  pre-sale pricing tools. Listings priced within ±5% of MMR are')
print(f'  categorised as "near market value" and are likely to attract')
print(f'  competitive bids without under-pricing the asset.')
print()
print('Recommendation 2: Vehicle Age and Mileage-Based Inventory Sourcing')
print(f'  Depreciation analysis shows that selling price drops significantly')
print(f'  with vehicle age. Vehicles aged 1-3 years represent the sweet spot')
print(f'  where the depreciation benefit from new-car pricing is captured')
print(f'  while resale values remain high. Fleet sourcing strategy should')
print(f'  prioritise low-mileage (below the dataset median of')
print(f'  {df["odometer"].median():,.0f} miles) vehicles under 3 years old,')
print(f'  as these consistently achieve above-average auction prices.')
print()
print('Recommendation 3: Model-Supported Price Reserve Setting')
print(f'  The ML regression model can be deployed as a price-floor tool at')
print(f'  auction. Before listing any vehicle, the model generates a predicted')
print(f'  price from vehicle features and MMR. Sellers can set the reserve')
print(f'  price at predicted price × 0.95 to minimise under-selling while')
print(f'  remaining competitive. Model B (without MMR, R²={metrics_B["R2"]:.4f})')
print(f'  can be used as a cross-check when MMR data is unavailable.')

## 16. Export Key Metrics for Node.js Dashboard

In [ ]:
# Compute all key metrics
key_metrics = {
    'total_records':        int(len(df)),
    'avg_selling_price':    round(float(df['sellingprice'].mean()), 2),
    'median_selling_price': round(float(df['sellingprice'].median()), 2),
    'avg_mmr':              round(float(df['mmr'].mean()), 2),
    'avg_condition':        round(float(df['condition'].mean()), 2),
    'avg_odometer':         round(float(df['odometer'].mean()), 2),
    'top_make':             str(df['make'].mode()[0]),
    'total_makes':          int(df['make'].nunique()),
    'date_range_start':     str(df['saledate_parsed'].min().date()),
    'date_range_end':       str(df['saledate_parsed'].max().date()),
    'model_A_MAE':          metrics_A['MAE'],
    'model_A_RMSE':         metrics_A['RMSE'],
    'model_A_R2':           metrics_A['R2'],
    'model_B_MAE':          metrics_B['MAE'],
    'model_B_RMSE':         metrics_B['RMSE'],
    'model_B_R2':           metrics_B['R2'],
    'model_name':           MODEL_NAME,
    'hypothesis_1_p':       float(kw_p),
    'hypothesis_2_rho':     float(sp_r),
    'hypothesis_2_p':       float(sp_p),
    'hypothesis_3_r':       float(pr_r),
    'hypothesis_3_p':       float(pr_p),
    'valuation_summary':    val_summary,
    'corr_mmr_price':       round(float(obs_corr_mmr), 4),
    'corr_odo_price':       round(float(obs_corr_odo), 4),
    'corr_age_price':       round(float(obs_corr_age), 4),
    'top10_make_avg_price': make_avg.to_dict(orient='records'),
    'age_price_table':      age_price.to_dict(orient='records')
}

for path in ['../outputs/metrics/key_metrics.json', '../Node.js_UI/public/key_metrics.json']:
    with open(path, 'w') as f:
        json.dump(key_metrics, f, indent=2, default=str)
print('Key metrics exported to outputs/metrics/key_metrics.json')
print()
print('=== FINAL DATASET STATS ===')
print(f'Clean records:           {len(df):,}')
print(f'Average selling price:   ${key_metrics["avg_selling_price"]:,.2f}')
print(f'Median selling price:    ${key_metrics["median_selling_price"]:,.2f}')
print(f'Average MMR:             ${key_metrics["avg_mmr"]:,.2f}')
print(f'Average odometer:        {key_metrics["avg_odometer"]:,.0f} miles')
print(f'Average condition score: {key_metrics["avg_condition"]:.2f}')
print(f'Model A R²:              {key_metrics["model_A_R2"]}')
print(f'Model A RMSE:            ${key_metrics["model_A_RMSE"]:,.2f}')
print()
print('All outputs saved. Notebook complete.')